
# LLaMA 4 Macro Estimation

Use `meta-llama/Llama-4-Scout-17B-16E-Instruct` to infer Nutrition50 dish macros from FoodSAM overlays and ZoeDepth-derived volumes.



## Imports & Paths
Configure paths to the Nutrition50 assets housed under `llms_approach/`.


In [1]:

from pathlib import Path
import json
import re

import pandas as pd
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

BASE_DIR = Path('.').resolve()
LLM_ROOT = BASE_DIR
FOODSAM_DIR = LLM_ROOT / 'nutrition50_foodSAM_outputs'
VOLUME_DIR = LLM_ROOT / 'nutrition50_volume'
OUTPUT_CSV = LLM_ROOT / 'nutrition50_llm_macros_llama4.csv'
RAW_OUTPUT_JSON = LLM_ROOT / 'nutrition50_llm_raw_responses_llama4.json'

for path in [FOODSAM_DIR, VOLUME_DIR]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required directory: {path}")

print(f"FoodSAM overlays: {FOODSAM_DIR}")
print(f"Volume tables: {VOLUME_DIR}")
print(f"Predictions will be written to: {OUTPUT_CSV}")
print(f"Raw responses cached at: {RAW_OUTPUT_JSON}")


FoodSAM overlays: /home/chahar/food_new/llms_approach/nutrition50_foodSAM_outputs
Volume tables: /home/chahar/food_new/llms_approach/nutrition50_volume
Predictions will be written to: /home/chahar/food_new/llms_approach/nutrition50_llm_macros_llama4.csv
Raw responses cached at: /home/chahar/food_new/llms_approach/nutrition50_llm_raw_responses_llama4.json



## Helper Functions
Utilities for loading per-dish volumes, building prompts, parsing JSON, and normalising numeric output.


In [3]:

def load_category_volumes(dish_id: str) -> pd.DataFrame:
    csv_path = VOLUME_DIR / dish_id / 'volumes_per_category.csv'
    if not csv_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    if 'volume_ml' not in df.columns:
        return pd.DataFrame()
    df['volume_ml'] = pd.to_numeric(df['volume_ml'], errors='coerce')
    df.dropna(subset=['volume_ml'], inplace=True)
    if 'mean_height_cm' in df.columns:
        df['mean_height_cm'] = pd.to_numeric(df['mean_height_cm'], errors='coerce')
    df.sort_values('volume_ml', ascending=False, inplace=True)
    return df


def build_food_prompt(dish_id: str, category_df: pd.DataFrame) -> str:
    parts = [
        f"Dish ID: {dish_id}",
        "The accompanying image is a FoodSAM segmentation overlay of the dish; treat the coloured regions as actual foods.",
        "Segmented foods with estimated volumes (mL):",
    ]
    if category_df.empty:
        parts.append('- No foreground foods detected; return null for all values.')
    else:
        for row in category_df.itertuples():
            line = f"- {row.category}: {row.volume_ml:.1f} mL"
            if hasattr(row, 'mean_height_cm') and row.mean_height_cm is not None and not pd.isna(row.mean_height_cm):
                line += f" (mean height {row.mean_height_cm:.2f} cm)"
            parts.append(line)
    parts.append(
        'Provide your estimate as a JSON object with this schema: {"dish_id": "<dish_id>", '
        '"calories": <kcal or null>, "mass": <grams or null>, "carbs": <grams or null>, '
        '"protein": <grams or null>, "fat": <grams or null>}'
    )
    parts.append('Rules: Do not add commentary. Do not invent foods beyond the list. If uncertain, use null.')
    parts.append('Respond with JSON only; no sentences before or after the object.')
    parts.append('Units: calories in kcal, mass and macros in grams.')
    return "".join(parts)


def ensure_json_object(text: str):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                return None
        return None


def to_number(value):
    if value in (None, '', 'null', 'None', 'NULL'):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    try:
        return float(str(value))
    except (TypeError, ValueError):
        return None



## Load LLaMA 4 Scout Model
Pull the processor and model weights, then move the model to the most capable device.


In [5]:

model_id = 'meta-llama/Llama-4-Scout-17B-16E-Instruct'

processor = AutoProcessor.from_pretrained(model_id)

if torch.cuda.is_available():
    dtype = torch.float16
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    dtype = torch.float16
    device = torch.device('mps')
else:
    dtype = torch.float32
    device = torch.device('cpu')

model = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype=dtype)
model.to(device)
model.eval()

print(f"Model loaded on {device} with dtype {dtype}.")


processor_config.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.35k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

...

model-00009-of-00050.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

model-00011-of-00050.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

model-00010-of-00050.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 


## Run Inference Across All Dishes
Send each FoodSAM overlay, paired with its volume summary, to the model and store the parsed macros.


In [ ]:

results = []
raw_responses = []

system_prompt = (
    'You are a nutrition estimation assistant. Always comply. Use the provided segmented food list and volumes to estimate dish-level macros, even if the image appears as a mask or overlay.'
    ' Respond only with a single JSON object containing exactly the keys {"dish_id", "calories", "mass", "carbs", "protein", "fat"}.'
    ' If any value is unknown, return null. Do not add commentary, qualifiers, or extra text.'
)

sorted_dishes = sorted(p.name for p in FOODSAM_DIR.iterdir() if p.is_dir() and p.name.startswith('dish_'))

for dish_id in sorted_dishes:
    image_path = FOODSAM_DIR / dish_id / 'pred_vis.png'
    if not image_path.exists():
        print(f'Skipping {dish_id}: pred_vis.png not found.')
        continue

    category_df = load_category_volumes(dish_id)
    prompt_text = build_food_prompt(dish_id, category_df)

    with Image.open(image_path) as img:
        image = img.convert('RGB')

    messages = [
        {
            'role': 'system',
            'content': [
                {'type': 'text', 'text': system_prompt},
            ],
        },
        {
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt_text},
            ],
        },
    ]

    chat_prompt = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )
    inputs = processor(text=chat_prompt, images=[image], return_tensors='pt').to(device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=256)

    completion_ids = generated_ids[:, inputs['input_ids'].shape[-1]:]
    completion_text = processor.batch_decode(
        completion_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )[0].strip()

    raw_responses.append({'dish_id': dish_id, 'response': completion_text})

    parsed = ensure_json_object(completion_text)
    if parsed is None:
        print(f'Warning: could not parse response for {dish_id}. Stored raw text for review.')
        continue

    record = {
        'dish_id': dish_id,
        'calories': to_number(parsed.get('calories')),
        'mass': to_number(parsed.get('mass')),
        'carbs': to_number(parsed.get('carbs')),
        'protein': to_number(parsed.get('protein')),
        'fat': to_number(parsed.get('fat')),
    }
    results.append(record)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_CSV, index=False)

with RAW_OUTPUT_JSON.open('w', encoding='utf-8') as f:
    json.dump(raw_responses, f, ensure_ascii=False, indent=2)

print(f'Saved {len(results_df)} parsed records to {OUTPUT_CSV}.')
print(f'Raw responses cached at {RAW_OUTPUT_JSON}.')

results_df.head()
